# 31 — Self-Supervised and Contrastive Learning for Medical Images

Medical datasets often contain many unlabeled images but relatively few high-quality labels.

Self-supervised learning (SSL) learns useful representations before the downstream supervised task.

We will study:

- Representation learning
- Positive and negative pairs
- Contrastive learning
- SimCLR intuition
- Augmentation design
- Encoder and projection head
- NT-Xent loss
- Linear probing
- Fine-tuning
- Ultrasound-specific SSL


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print("PyTorch:",torch.__version__)


# 1. Self-Supervised Representation Learning

Learn:

$$
f(x)=z
$$

without manual class labels, then reuse $f$ for a downstream task.


# 2. Contrastive Learning

Two augmented views of the same image are a positive pair. Other images often act as negatives.


In [ ]:
def toy_image(seed=0,size=32):
    g=torch.Generator().manual_seed(seed)
    x=torch.randn(1,size,size,generator=g)*0.08
    x[:,8:24,14:18]+=1
    return x.clamp(0,1)

def augment(x):
    shift=int(torch.randint(-2,3,(1,)).item())
    y=torch.roll(x,shift,2)
    y=(y*(0.9+0.2*torch.rand(1).item())).clamp(0,1)
    return (y+torch.randn_like(y)*0.03).clamp(0,1)


# 3. Augmentation Defines Invariance

If two transformations are treated as equivalent views, the model is encouraged to ignore their differences.

For ultrasound, augmentation must preserve clinical meaning.


In [ ]:
x=toy_image()
v1=augment(x);v2=augment(x)
fig,axes=plt.subplots(1,3,figsize=(8,3))
for a,z,t in zip(axes,[x,v1,v2],["Original","View 1","View 2"]):
    a.imshow(z.squeeze(),cmap="gray");a.set_title(t);a.axis("off")
plt.show()


# 4. Encoder


In [ ]:
class SSLEncoder(nn.Module):
    def __init__(self,dim=128):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)
        )
        self.fc=nn.Linear(64,dim)
    def forward(self,x):
        return self.fc(torch.flatten(self.net(x),1))


# 5. Projection Head

SimCLR applies contrastive loss to:

$$
z=g(f(x))
$$

not directly to the downstream representation $f(x)$.


In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self,cin=128,cout=64):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(cin,128),nn.ReLU(),nn.Linear(128,cout))
    def forward(self,x): return self.net(x)


# 6. Cosine Similarity


In [ ]:
def cosine_matrix(z):
    z=F.normalize(z,dim=1)
    return z@z.T


# 7. NT-Xent Loss

For anchor $i$ and positive $j$:

$$
\ell_{i,j}
=
-\log
\frac{\exp(sim(z_i,z_j)/\tau)}
{\sum_{k\neq i}\exp(sim(z_i,z_k)/\tau)}
$$


In [ ]:
def nt_xent(z1,z2,temperature=0.2):
    n=z1.size(0)
    z1=F.normalize(z1,dim=1);z2=F.normalize(z2,dim=1)
    z=torch.cat([z1,z2],dim=0)
    sim=z@z.T/temperature
    eye=torch.eye(2*n,dtype=torch.bool,device=z.device)
    sim=sim.masked_fill(eye,-1e9)
    targets=torch.cat([
        torch.arange(n,device=z.device)+n,
        torch.arange(n,device=z.device)
    ])
    return F.cross_entropy(sim,targets)


In [ ]:
z1=torch.randn(16,64)
z2=z1+0.1*torch.randn(16,64)
print(nt_xent(z1,z2).item())


# 8. Full SimCLR-Style Model


In [ ]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=SSLEncoder(128)
        self.projector=ProjectionHead(128,64)
    def forward(self,x):
        h=self.encoder(x)
        return h,self.projector(h)


# 9. SSL Training Step


In [ ]:
def ssl_step(model,batch,optimizer):
    a=torch.stack([augment(x) for x in batch])
    b=torch.stack([augment(x) for x in batch])
    _,z1=model(a);_,z2=model(b)
    loss=nt_xent(z1,z2)
    optimizer.zero_grad(set_to_none=True)
    loss.backward();optimizer.step()
    return loss.item()


# 10. Linear Probing

Freeze encoder; train only a linear classifier. This tests how useful the representation is.


In [ ]:
class LinearProbe(nn.Module):
    def __init__(self,encoder,dim=128,num_classes=3):
        super().__init__()
        self.encoder=encoder
        for p in self.encoder.parameters(): p.requires_grad=False
        self.head=nn.Linear(dim,num_classes)
    def forward(self,x):
        with torch.no_grad():
            h=self.encoder(x)
        return self.head(h)


# 11. Fine-Tuning

After probing, unfreeze some/all encoder layers and fine-tune with a smaller learning rate.


# 12. Why SSL Can Help Medical Imaging

Useful when:

- Labels are expensive
- Large unlabeled archives exist
- Domain-specific features differ from ImageNet


# 13. Ultrasound Augmentation Caution

Do not automatically use aggressive flips/crops/rotations. The augmentation defines what the model should treat as equivalent.


# 14. False Negatives

Two different patients can have similar pathology. Instance contrastive learning may incorrectly push them apart.


# 15. Positive Pair Choices

Possible ultrasound positives:

- Two augmentations of same frame
- Nearby cine frames
- Multiple views from same study

Each choice creates a different invariance assumption.


# 16. Alternatives to Large-Negative Contrastive Learning

Examples:

- BYOL
- SimSiam
- DINO-style self-distillation

These can learn without classic large negative sets.


# 17. Patient Leakage in SSL

Even unlabeled pretraining requires a clearly defined protocol. Decide whether test-patient images are permitted. For strict inductive evaluation, keep them excluded.


# 18. Linear Probe vs Fine-Tuning

Linear probe evaluates representation quality with a frozen encoder.

Fine-tuning evaluates end-task adaptability.


# 19. Label-Efficiency Experiment

Compare pretraining methods using:

- 10% labels
- 25%
- 50%
- 100%

This directly tests whether SSL helps when labels are scarce.


# 20. Fair Comparison

Compare:

1. Random initialization
2. ImageNet pretraining
3. Ultrasound SSL pretraining

using identical patient folds, seeds, and fine-tuning schedules.


# 21. Common Mistakes

- Unrealistic augmentation
- Test-patient leakage during SSL
- Comparing different downstream protocols
- Evaluating only contrastive loss
- Claiming SSL benefit without supervised baselines


# 22. Exercises

1. Create positive pairs.
2. Normalize features.
3. Compute cosine similarity.
4. Implement NT-Xent.
5. Build encoder/projector.
6. Linear probe.
7. Fine-tune encoder.
8. Design ultrasound augmentations.
9. Explain false negatives.
10. Design label-efficiency analysis.


# 23. Key Takeaways

SSL flow:

$$
\boxed{
Unlabeled\ Image
\rightarrow
Augmented\ Views
\rightarrow
Encoder
\rightarrow
Representation
}
$$

The most important ultrasound design decision is often the augmentation policy.


# Next Notebook

# 32 — Uncertainty, Calibration, Ensembling, and Selective Prediction

In the next notebook, we will study calibration, temperature scaling, uncertainty, ensembles, MC dropout, and abstention.
